In [4]:
import pandas as pd

In [5]:
# 1. Loading individual SimScale result CSVs exported from parametric run
print("Loading individual SimScale CSV exports...")
ux_inlet = pd.read_csv('SimScale Results/Ux.csv')
ux_outlet = pd.read_csv('SimScale Results/Ux (1).csv')
p_inlet = pd.read_csv('SimScale Results/p.csv')
p_outlet = pd.read_csv('SimScale Results/p (1).csv')
k_inlet = pd.read_csv('SimScale Results/k.csv')
k_outlet = pd.read_csv('SimScale Results/k (1).csv')
omega_inlet = pd.read_csv('SimScale Results/omega.csv')
omega_outlet = pd.read_csv('SimScale Results/omega (1).csv')
nut_inlet = pd.read_csv('SimScale Results/nut.csv')
nut_outlet = pd.read_csv('SimScale Results/nut (1).csv')

Loading individual SimScale CSV exports...


In [6]:
# 2. Merging all dataframes sequentially on the 'Run #' column
master_df = (
    ux_inlet[['Run #', 'face33@Body4']]
    .rename(columns={'face33@Body4': 'Test_Velocity_V_test_ms'})
    .merge(
        p_inlet[['Run #', 'face33@Body4']].rename(
            columns={'face33@Body4': 'Pressure_Face33_Pa'}
        ),
        on='Run #',
    )
    .merge(
        p_outlet[['Run #', 'face71@Body4']].rename(
            columns={'face71@Body4': 'Pressure_Face71_Pa'}
        ),
        on='Run #',
    )
    .merge(
        k_inlet[['Run #', 'face33@Body4']].rename(
            columns={'face33@Body4': 'TKE_k_Face33'}
        ),
        on='Run #',
    )
    .merge(
        k_outlet[['Run #', 'face71@Body4']].rename(
            columns={'face71@Body4': 'TKE_k_Face71'}
        ),
        on='Run #',
    )
    .merge(
        omega_inlet[['Run #', 'face33@Body4']].rename(
            columns={'face33@Body4': 'Omega_Face33'}
        ),
        on='Run #',
    )
    .merge(
        omega_outlet[['Run #', 'face71@Body4']].rename(
            columns={'face71@Body4': 'Omega_Face71'}
        ),
        on='Run #',
    )
    .merge(
        nut_inlet[['Run #', 'face33@Body4']].rename(
            columns={'face33@Body4': 'Turb_Viscosity_nut_Face33'}
        ),
        on='Run #',
    )
    .merge(
        nut_outlet[['Run #', 'face71@Body4']].rename(
            columns={'face71@Body4': 'Turb_Viscosity_nut_Face71'}
        ),
        on='Run #',
    )
)

In [7]:
# 3. Adding a calculated Pressure Drop column (Face 33 minus Face 71)
master_df['Pressure_Drop_Pa'] = (
    master_df['Pressure_Face33_Pa'] - master_df['Pressure_Face71_Pa']
)

# Reordering columns for better readability
cols = ['Run #', 'Test_Velocity_V_test_ms', 'Pressure_Face33_Pa', 'Pressure_Face71_Pa', 'Pressure_Drop_Pa']
other_cols = [c for c in master_df.columns if c not in cols]
master_df = master_df[cols + other_cols]

# 4. Saving to a single CSV file
output_filename = 'simscale_combined_results.csv'
master_df.to_csv(output_filename, index=False)

print(f"\nSuccessfully generated '{output_filename}'!")
print(f"Total rows (runs): {len(master_df)}")
print("\nPreview of merged data:")
print(master_df.head())


Successfully generated 'simscale_combined_results.csv'!
Total rows (runs): 26

Preview of merged data:
   Run #  Test_Velocity_V_test_ms  Pressure_Face33_Pa  Pressure_Face71_Pa  \
0    1.0                      5.0           26.005606          -14.535633   
1    2.0                      6.0           35.813699          -21.914046   
2    3.0                      7.0           48.780539          -29.000997   
3    4.0                      8.0           61.840775          -38.944477   
4    5.0                      9.0           74.419233          -52.334814   

   Pressure_Drop_Pa  TKE_k_Face33  TKE_k_Face71  Omega_Face33  Omega_Face71  \
0         40.541238       0.09375      0.057390     59.894678  34284.882256   
1         57.727745       0.13500      0.101388     71.873614  34284.882256   
2         77.781537       0.18375      0.160053     83.852549  34284.882256   
3        100.785252       0.24000      0.234346     95.831485  34284.882256   
4        126.754046       0.30375     